<a href="https://colab.research.google.com/github/roxanamore280-sketch/SQL_librerias_ejercicio/blob/main/Pontia_SuperStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_excel('PontiaSuperStore.xls', sheet_name="Orders")
df.head(3)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [29]:
#Sales/Quantity = precio de venta por unidad

df['Sales/Quantity'] = df['Sales'] / df['Quantity']

#Profit/Quantity = beneficio por unidad

df['Profit/Quantity'] = df['Profit'] / df['Quantity']

#Flag_descuento "Con descuento" si hubo descuento si no "Sin descuento"

df['flag_descuento'] = np.where(df['Discount'] > 0, 'Con descuento', 'Sin descuento')

#Year año extraido del Order Date

df['Year'] = df['Order Date'].dt.year






In [30]:
df[['Sales', 'Quantity', 'Sales/Quantity', 'Discount', 'flag_descuento', 'Order Date', 'Year']].head()

,Sales,Quantity,Sales/Quantity,Discount,flag_descuento,Order Date,Year
0,261.9600,2,130.9800,0.00,Sin descuento,2016-11-08,2016
1,731.9400,3,243.9800,0.00,Sin descuento,2016-11-08,2016
2,14.6200,2,7.3100,0.00,Sin descuento,2016-06-12,2016
3,957.5775,5,191.5155,0.45,Con descuento,2015-10-11,2015
4,22.3680,2,11.1840,0.20,Con descuento,2015-10-11,2015


In [31]:
resumen = df.groupby('flag_descuento', as_index=False)['Profit'].sum()
resumen

,flag_descuento,Profit
0,Con descuento,-34590.5815
1,Sin descuento,321046.5414


In [33]:
fig1 = px.bar(
    resumen,
    x='flag_descuento',
    y='Profit',
    color='flag_descuento',
    title='Beneficio total: pedidos con decuento vs sin descuento',
    labels={'Profit': 'Beneficio total (€)', 'flag_decuento': ''},
    text_auto='.2s'
)
fig1.show()

In [37]:
df['margen'] = df['Profit'] / df['Sales']

margen_desc = df.groupby('Discount', as_index=False)['margen'].mean()
margen_desc

,Discount,margen
0,0.00,0.340179
1,0.10,0.155792
2,0.15,0.034163
3,0.20,0.176839
4,0.30,-0.115481
5,0.32,-0.174292
6,0.40,-0.222492
7,0.45,-0.454545
8,0.50,-0.549091
9,0.60,-0.689130


In [38]:
fig2 = px.line(
    margen_desc,
    x='Discount',
    y='margen',
    markers=True,
    title='Margen medio segun el nivel de descuento aplicado',
    labels={'Discount': 'Descuento aplicado', 'margen': 'Margen medio (Profit/Sales)'}
    )

#linea horizontal en 0: por debajo de aquí se pierde dinero

fig2.add_hline(y=0, line_dash='dash', line_color='red')
fig2.show()


In [40]:
por_producto = df.groupby(
    ['Category', 'Sub-Category', 'flag_descuento'],
    as_index=False
)['Profit'].sum()
por_producto



,Category,Sub-Category,flag_descuento,Profit
0,Furniture,Bookcases,Con descuento,-9548.2677
1,Furniture,Bookcases,Sin descuento,6075.7117
2,Furniture,Chairs,Con descuento,4657.0702
3,Furniture,Chairs,Sin descuento,21933.0961
4,Furniture,Furnishings,Con descuento,-3788.8253
5,Furniture,Furnishings,Sin descuento,16847.9689
6,Furniture,Tables,Con descuento,-31001.7808
7,Furniture,Tables,Sin descuento,13276.2997
8,Office Supplies,Appliances,Con descuento,-5045.7307
9,Office Supplies,Appliances,Sin descuento,23183.7361


In [41]:
fig3 = px.bar(
    por_producto,
    x='Profit',
    y='Sub-Category',
    color='flag_descuento',
    facet_col='Category',
    title='Beneficio por sub-categoria, según tenga descuento o no',
    labels={'Profit': 'Beneficio total (€)', 'Sub-Category':'', 'Flag_descuento': ''},
    orientation='h',
  )

fig3.update_yaxes(matches=None) # cada categoria muestra solo sub-categoria
fig3.show()